# SoFIFA cleaning (revised) — line‑by‑line explained

This notebook is a **minimal deviation** from the previously delivered `SoFIFA_cleaning_revised.ipynb`, but it now explains what each line is doing.

**Goal (same intent as your original):**
1. Load the raw SoFIFA dataset (`player_stats.csv`)
2. Validate that the file is not “column‑shift corrupted”
3. Clean types + engineer a few practical features
4. Deduplicate to one row per `player_id` (keeping the latest `version`)
5. Save to **Parquet** (and optionally a small CSV preview)

---

## Why the “repair reader” exists (important)

In your earlier pipeline, a common real-world failure is:

- Some text fields contain commas (e.g., “play styles”, “specialities”).
- If the export/scrape did **not** quote those text fields correctly, a CSV parser treats those commas as separators.
- That shifts columns to the right, so numeric columns can end up containing URLs (e.g., `https://sofifa.com/player/...`).
- When you then coerce to numeric, you silently get `NaN` and lose the signal.

So we add a **defensive loader** that can realign rows using the `url` token as an anchor.


## 0) Imports (each line explained)

In [ ]:
# os: allows us to check if files exist and to work with file paths
import os

# re: regular expressions; used here for simple string-pattern checks
import re

# math: basic math utilities (we only need it occasionally; safe to include)
import math

# pandas: main data tool for tables (DataFrames)
import pandas as pd

# numpy: numeric arrays + NaN handling + vectorised operations
import numpy as np

# date: for simple calendar dates (used for reference dates if needed)
from datetime import date

# Make pandas show more columns when printing (purely for convenience)
pd.set_option("display.max_columns", 200)

# Make pandas use a wider console width (purely for convenience)
pd.set_option("display.width", 140)


## 1) Configuration (each line explained)

In [ ]:
# Path to your raw SoFIFA export/scrape CSV
RAW_CSV_PATH = "/mnt/data/player_stats.csv"

# Primary output path (Parquet preserves types; best for ML pipelines)
OUT_PARQUET_PATH = "/mnt/data/player_stats_cleaned.parquet"

# Optional CSV output path (useful for quick viewing; types are less stable than Parquet)
OUT_CSV_PATH = "/mnt/data/player_stats_cleaned.csv"

# If not None, only the first N rows are written to CSV (keeps CSV smaller/faster)
# Set to None if you truly want the full CSV written
CSV_PREVIEW_N = 5000

# Business rule: cap contract years left (prevents extreme outliers and nonsense values)
CONTRACT_YEARS_CAP = 10

# If True, use our URL-anchored repair reader; if False, do a normal pd.read_csv
# Keep this True if you suspect unquoted commas caused misalignment in the raw file
USE_REPAIR_READER = True


## 2) Robust CSV reader with URL‑anchor repair (each line explained)

In [ ]:
def read_csv_with_url_anchor_repair(
    csv_path: str,
    encoding: str = "utf-8",
    expected_url_col: str = "url",
    url_prefix: str = "https://sofifa.com/player/",
) -> pd.DataFrame:
    """Read a CSV that may be column-shift corrupted by unquoted commas in text fields.

    High-level idea:
    - Read the header to know the expected column order and count.
    - For each subsequent line:
        - split by ',' (naive)
        - find a token that looks like a SoFIFA player URL (anchor)
        - force that token to land in the expected 'url' column position
        - pad/truncate tokens to match the header length
    """

    # If the file doesn't exist, stop immediately with a clear error
    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"File not found: {csv_path}")

    # Open the CSV file as plain text so we can manually parse lines
    with open(csv_path, "r", encoding=encoding, errors="replace") as f:
        # Read the first line (header) and remove the trailing newline
        header_line = f.readline().rstrip("\n")

        # Split header by commas to get the column names
        columns = header_line.split(",")

        # Number of columns we expect each row to have
        ncols = len(columns)

        # Ensure the URL column exists (we use it as an anchor for repair)
        if expected_url_col not in columns:
            raise ValueError(
                f"Expected column '{expected_url_col}' not found in header. "
                f"Columns found: {columns[:10]} ... ({ncols} total)"
            )

        # Index where 'url' SHOULD be according to the header
        url_idx_expected = columns.index(expected_url_col)

        # We'll store repaired rows here (as lists of strings)
        rows = []

        # Counters to report quality of the parsing
        bad_lines = 0       # lines that still end with wrong column count
        no_url_lines = 0    # lines where we couldn't find any URL token at all

        # Enumerate over each remaining line in the file
        # start=2 because we already read line 1 as the header
        for line_num, line in enumerate(f, start=2):
            # Remove newline at the end of the line
            line = line.rstrip("\n")

            # Skip completely empty lines
            if not line.strip():
                continue

            # Naively split the line by commas (this is where CSV corruption happens)
            tokens = line.split(",")

            # Find positions where a token starts with the SoFIFA player URL prefix
            url_positions = [i for i, tok in enumerate(tokens) if tok.startswith(url_prefix)]

            # If we can't find any URL token, we cannot anchor-repair; we just pad/truncate
            if not url_positions:
                no_url_lines += 1

                # If too few columns, pad with blanks
                if len(tokens) < ncols:
                    tokens = tokens + [""] * (ncols - len(tokens))

                # If too many columns, merge the overflow into the last column
                elif len(tokens) > ncols:
                    tokens = tokens[:ncols-1] + [",".join(tokens[ncols-1:])]

                # Save row tokens
                rows.append(tokens)
                continue

            # If multiple URL-like tokens exist, keep the last one (most likely the real url)
            url_idx_found = url_positions[-1]

            # CASE 1: URL appears too early -> insert blank columns right before it
            if url_idx_found < url_idx_expected:
                # How many blanks do we need to insert?
                insert_count = url_idx_expected - url_idx_found

                # Insert that many blank tokens before the URL token
                tokens = tokens[:url_idx_found] + [""] * insert_count + tokens[url_idx_found:]

                # After insertion, the URL token is now at the expected index
                url_idx_found = url_idx_expected

            # CASE 2: URL appears too late -> merge the extra tokens back into the previous field
            # This typically happens when a text column before url had unquoted commas
            if url_idx_found > url_idx_expected:
                # Surplus tokens between expected url position and the found url position
                surplus = tokens[url_idx_expected:url_idx_found]

                # Merge surplus into the column immediately before the expected url column
                # (because the final text column before url is usually what got split)
                if url_idx_expected - 1 >= 0:
                    left = tokens[url_idx_expected - 1] or ""
                    glue = "," if left else ""
                    tokens[url_idx_expected - 1] = left + glue + ",".join(surplus)

                    # Remove the surplus tokens from the list
                    del tokens[url_idx_expected:url_idx_found]

                # Re-locate the URL token index after modifications
                # (Find the first token that starts with url_prefix)
                url_idx_found = next(i for i, tok in enumerate(tokens) if tok.startswith(url_prefix))

            # Final length correction: every row MUST match ncols
            if len(tokens) < ncols:
                tokens = tokens + [""] * (ncols - len(tokens))
            elif len(tokens) > ncols:
                tokens = tokens[:ncols-1] + [",".join(tokens[ncols-1:])]

            # Record if the line still didn't land exactly on the correct length
            if len(tokens) != ncols:
                bad_lines += 1

            # Save the repaired tokens
            rows.append(tokens)

    # Create a DataFrame from the repaired rows using the header columns
    df = pd.DataFrame(rows, columns=columns)

    # Print a small loader report (helps debugging)
    print(f"Loaded rows: {len(df):,}")
    print(f"Header columns: {ncols}")
    print(f"Lines without URL anchor: {no_url_lines:,}")
    if bad_lines:
        print(f"WARNING: {bad_lines:,} lines still had unexpected column counts after repair.")

    # Return the parsed DataFrame
    return df


def read_raw_sofifa(csv_path: str, use_repair_reader: bool = True) -> pd.DataFrame:
    """Choose a reader based on the config flag."""

    # If we want the robust reader, use the repair function above
    if use_repair_reader:
        return read_csv_with_url_anchor_repair(csv_path)

    # Otherwise, use normal pandas CSV parsing (only safe if the file is well-formed)
    return pd.read_csv(csv_path, low_memory=False)


## 3) Load data + first‑pass validation (each line explained)

In [ ]:
# Load the raw dataset (using repair reader if enabled)
df_raw = read_raw_sofifa(RAW_CSV_PATH, use_repair_reader=USE_REPAIR_READER)

# Print the shape (rows, columns) so we know what we loaded
print(df_raw.shape)

# Show the first 3 rows to visually sanity check
display(df_raw.head(3))


In [ ]:
# Define columns we consider essential for downstream work (IDs, versioning, basic identity, URL)
required_cols = ["player_id", "version", "name", "positions", "dob", "url"]

# Check which required columns are missing
missing = [c for c in required_cols if c not in df_raw.columns]

# If any are missing, stop immediately
if missing:
    raise ValueError(f"Missing required columns: {missing}")

# Pick a few columns that should be numeric (if they exist in your dataset)
# These are used as "sentinel" corruption checks.
numeric_sentinel_cols = [
    c for c in ["overall_rating", "potential", "defending_standing_tackle", "goalkeeping_gk_diving"]
    if c in df_raw.columns
]

def frac_looks_like_url(s: pd.Series) -> float:
    """Return fraction of values that contain a SoFIFA player URL substring."""
    s = s.astype(str)
    return s.str.contains("sofifa.com/player", na=False).mean()

# Print the fraction of URL-like strings in numeric-sentinel columns
for c in numeric_sentinel_cols:
    frac = frac_looks_like_url(df_raw[c])
    print(f"{c}: {frac:.3%} of values look like URLs")

# If any sentinel has >1% URL-like values, warn that misalignment likely exists
high_url_cols = [c for c in numeric_sentinel_cols if frac_looks_like_url(df_raw[c]) > 0.01]
if high_url_cols:
    print("\nWARNING: Possible misalignment detected (URL-like strings in numeric columns):", high_url_cols)
    print("Suggestion: keep USE_REPAIR_READER=True or (best) re-export the CSV with correct quoting or as Parquet.")


## 4) Cleaning + feature engineering (each line explained)

In [ ]:
# Work on a copy so df_raw remains untouched (useful for debugging)
df = df_raw.copy()

# Replace common empty-string placeholders with NaN so pandas treats them as missing values
df = df.replace({"": np.nan, "None": np.nan, "nan": np.nan})

def coerce_numeric_if_mostly_numeric(df: pd.DataFrame, threshold: float = 0.85) -> pd.DataFrame:
    """Convert object columns to numeric if they look numeric for most non-null values."""
    out = df.copy()

    for col in out.columns:
        if out[col].dtype == "object":
            s = out[col]

            # Skip known text fields to avoid converting names/URLs/etc.
            if col in {
                "name", "full_name", "description", "image",
                "positions", "play_styles", "specialities",
                "url", "country_name", "club_name", "league_name"
            }:
                continue

            # Sample non-null values
            sample = s.dropna().astype(str).head(2000)
            if sample.empty:
                continue

            # Share of values that look numeric
            numeric_like = sample.str.match(r"^-?\d+(\.\d+)?$").mean()

            # Coerce if mostly numeric
            if numeric_like >= threshold:
                out[col] = pd.to_numeric(out[col], errors="coerce")

    return out

# Apply numeric coercion
df = coerce_numeric_if_mostly_numeric(df)

# Parse date of birth into datetime
df["dob"] = pd.to_datetime(df["dob"], errors="coerce")

# Contract parsing
if "club_contract_valid_until" in df.columns:
    tmp_year = pd.to_numeric(df["club_contract_valid_until"], errors="coerce")
    tmp_date = pd.to_datetime(df["club_contract_valid_until"], errors="coerce")
    df["club_contract_valid_until_year"] = tmp_year
    df.loc[df["club_contract_valid_until_year"].isna() & tmp_date.notna(), "club_contract_valid_until_year"] = tmp_date.dt.year
else:
    df["club_contract_valid_until_year"] = np.nan

# Ensure version numeric
df["version"] = pd.to_numeric(df["version"], errors="coerce")

def infer_version_year(v):
    if pd.isna(v):
        return np.nan
    v = int(v)
    yy = int(str(v)[:2])
    return 2000 + yy if yy <= 79 else 1900 + yy

df["version_year"] = df["version"].apply(infer_version_year)

def age_on_july1(dob, year):
    if pd.isna(dob) or pd.isna(year):
        return np.nan
    ref = pd.Timestamp(year=int(year), month=7, day=1)
    return (ref - dob).days / 365.25

df["age_at_version"] = [age_on_july1(d, y) for d, y in zip(df["dob"], df["version_year"])]

df["contract_years_left"] = df["club_contract_valid_until_year"] - df["version_year"]
df.loc[df["contract_years_left"] < 0, "contract_years_left"] = 0
df["contract_years_left"] = df["contract_years_left"].clip(upper=CONTRACT_YEARS_CAP)
df["contract_unknown"] = df["club_contract_valid_until_year"].isna().astype(int)

def split_positions(x):
    if pd.isna(x):
        return []
    return [p.strip() for p in str(x).split(",") if p.strip()]

df["positions_list"] = df["positions"].apply(split_positions)
df["primary_position"] = df["positions_list"].apply(lambda lst: lst[0] if len(lst) else np.nan)

ALL_POS = ["GK","CB","LB","RB","LWB","RWB","CDM","CM","CAM","LM","RM","LW","RW","CF","ST"]
for p in ALL_POS:
    df[f"is_{p}"] = df["positions_list"].apply(lambda lst, p=p: int(p in lst))

for money_col in ["value", "wage", "release_clause"]:
    if money_col in df.columns:
        df[money_col] = pd.to_numeric(df[money_col], errors="coerce")
        df[f"log1p_{money_col}"] = np.log1p(df[money_col])

def mean_of_existing(cols):
    existing = [c for c in cols if c in df.columns]
    if not existing:
        return None
    return df[existing].mean(axis=1)

df["agg_pace"] = mean_of_existing(["movement_acceleration", "movement_sprint_speed"])
df["agg_shooting"] = mean_of_existing(["attacking_finishing", "power_shot_power", "skill_fk_accuracy", "attacking_volleys", "mentality_penalties"])
df["agg_passing"] = mean_of_existing(["attacking_short_passing", "skill_long_passing", "skill_curve", "skill_crossing", "skill_vision"])
df["agg_dribbling"] = mean_of_existing(["skill_dribbling", "skill_ball_control", "movement_agility", "movement_balance", "movement_reactions"])
df["agg_defending"] = mean_of_existing(["mentality_interceptions", "defending_marking_awareness", "defending_standing_tackle", "defending_sliding_tackle"])
df["agg_physical"] = mean_of_existing(["power_strength", "power_stamina", "power_jumping"])

print("Post-cleaning shape:", df.shape)
display(df.head(3))


## 5) Deduplication (each line explained)

In [ ]:
df["player_id"] = pd.to_numeric(df["player_id"], errors="coerce").astype("Int64")
df_sorted = df.sort_values(by=["player_id", "version"], ascending=[True, False])

before = len(df_sorted)
df_dedup = df_sorted.drop_duplicates(subset=["player_id"], keep="first").copy()
after = len(df_dedup)

print(f"Rows before dedupe: {before:,}")
print(f"Rows after dedupe : {after:,}")
print(f"Removed          : {before-after:,} duplicates")


## 6) Final validation report (each line explained)

In [ ]:
def missingness_report(df: pd.DataFrame, top_n: int = 25) -> pd.DataFrame:
    miss = df.isna().mean().sort_values(ascending=False)
    return miss.head(top_n).to_frame("missing_fraction")

report = missingness_report(df_dedup, top_n=30)
display(report)

assert df_dedup["player_id"].isna().mean() < 0.01, "Too many missing player_id values; something is wrong upstream."

url_ok = df_dedup["url"].astype(str).str.startswith("https://sofifa.com/player/").mean()
print(f"URL field looks valid in {url_ok:.2%} of rows")


## 7) Output (each line explained)

In [ ]:
df_dedup.to_parquet(OUT_PARQUET_PATH, index=False)
print("Wrote:", OUT_PARQUET_PATH)

if CSV_PREVIEW_N is None:
    df_dedup.to_csv(OUT_CSV_PATH, index=False)
    print("Wrote:", OUT_CSV_PATH)
else:
    df_dedup.head(CSV_PREVIEW_N).to_csv(OUT_CSV_PATH, index=False)
    print(f"Wrote preview ({CSV_PREVIEW_N:,} rows):", OUT_CSV_PATH)
